<header>
   <p  style='font-size:36px;font-family:Arial; color:#F0F0F0; background-color: #00233c; padding-left: 20pt; padding-top: 20pt;padding-bottom: 10pt; padding-right: 20pt;'>
       Vantage In-database Graph Analysis: Retail Products Recommendation
  <br>
       <img id="teradata-logo" src="https://storage.googleapis.com/clearscape_analytics_demo_data/DEMO_Logo/teradata.svg" alt="Teradata" style="width: 125px; height: auto; margin-top: 20pt;">
    </p>
</header>

<hr>

In [ ]:
# Teradata python package
from teradataml import *
from TeradataGE import td_graph_function, configure

import pandas as pd
import getpass
import numpy as np

# for local display only
import matplotlib.pyplot as plt
from matplotlib import cm, colors


In [ ]:
configure.graph_install_location = "GraphLib"

<hr>
<p style = 'font-size:28px;font-family:Arial;color:#00233C'><b>Connect to Teradata Vantage Instance</b></p>

In [ ]:
hostname = "xx.xx.xx.xx"
user = input(prompt=f"Username for database:")
password = getpass.getpass(prompt=f"Database password for {user}:")
databasename = "graphdb"
logmech = "TD2"

eng = create_context(host = hostname, username = user, password = password, database= databasename, logmech = logmech)
print(eng)

<hr>
<p style = 'font-size:28px;font-family:Arial;color:#00233C'><b>Market Basket Analysis (Assocation)</b></p>

In [ ]:
assoc_out_table = "order_product_assoc"


<hr>
<p style = 'font-size:34px;font-family:Arial;color:#00233C'><b>Neighbor & Traversal-Based Recommendations</b></p>

<hr>
<p style = 'font-size:28px;font-family:Arial;color:#00233C'><b>Node and Edge information</b></p>

In [ ]:
node_table = "products_v"
node_id = "product_id"
edge_table = "order_product_edge"
edge_from_id = "ITEM1OF2"
edge_to_id = "ITEM2OF2"

min_prob = 0.2
max_level = 5

In [ ]:
# Define both edge and node metadata
graph_obj = td_graph_function.td_graph_object(edge_table_name = edge_table,
                                              edge_from_node_column_name = edge_from_id,
                                              edge_to_node_column_name = edge_to_id,
                                              edge_weight_column_name = "CONFIDENCE",
                                              weight_type = 'P',
                                              node_table_name = node_table,
                                              node_id_column_name = node_id,
                                              node_label_column_name = 'product_name'
                                             )


In [ ]:
# Display the defined informartion
print(graph_obj.edge_info())
print(graph_obj.node_info())

<hr>
<p style = 'font-size:28px;font-family:Arial;color:#00233C'><b>Adding 1st item into the shopping cart</b></p>

In [ ]:
# Filter: Start with 1st sku added into cart
source_df = graph_obj.set_node_id_from_label_source("Strawberry Banana Yogurt")
source_df

In [ ]:
# Recommended Products
min_prob = 0.1
prod_recomm_df = graph_obj.td_topology(return_data = 'P', 
                                       weight_filter = f">{min_prob}" ,
                                       max_path_length = 5, 
                                       output_table = "prod_recommod_vt2",
                                       temp_output_table = True,
                                       show_query=False)
path_label_df, fig = graph_obj.td_graph_path_decode('prod_recommod_vt2', edge_attributes= 'dept_dept', include_plot=True)


In [ ]:
fig.show()

<hr>
<p style = 'font-size:28px;font-family:Arial;color:#00233C'><b>Adding 2nd item into the shopping cart</b></p>

In [ ]:
# Filter: Start with 1st sku added into cart
id = graph_obj.set_node_id_from_label_source("Vanilla Whole Milk Yogurt", if_exists='append')
id

In [ ]:
# Recommended Products 
prod_recomm_df = graph_obj.td_topology(return_data = 'P', 
                                       weight_filter = f">{min_prob}" ,
                                       max_path_length = 5, 
                                       output_table = "prod_recommod_vt2",
                                       temp_output_table = True,
                                       show_query=True)
path_label_df, fig = graph_obj.td_graph_path_decode('prod_recommod_vt2', edge_attributes= 'dept_dept', include_plot=True)

In [ ]:
fig.show()

<hr>
<p style = 'font-size:28px;font-family:Arial;color:#00233C'><b>What happend if my 2st item is not any recommended products before</b></p>

In [ ]:
# Filter: Start with 1st sku added into cart
id = graph_obj.set_node_id_from_label_source("Strawberry Banana Yogurt")
#id = graph_obj.set_node_id_from_label_source("2% Milk", show_query=True )
id = graph_obj.set_node_id_from_label_source("Organic Baby Spinach", if_exists = 'append')
id

In [ ]:
prod_recomm_df = graph_obj.td_topology(return_data = 'P', 
                                       weight_filter = f">{min_prob}" ,
                                       max_path_length = 5, 
                                       output_table = "prod_recommod_vt2",
                                       temp_output_table = True)
path_label_df, fig = graph_obj.td_graph_path_decode('prod_recommod_vt2', edge_attributes= 'dept_dept', include_plot=True)
fig

<hr>
<p style = 'font-size:34px;font-family:Arial;color:#00233C'><b>Community Recommendation</b></p>

In [ ]:
node_table = "products_v"
node_id = "product_id"
edge_table = "order_product_edge"
edge_from_id = "ITEM1OF2"
edge_to_id = "ITEM2OF2"

min_prob = 0.2
max_level = 5

In [ ]:
# pick the high confidence pairs
df = DataFrame(edge_table)
df = df[df.CONFIDENCE>=min_prob]
copy_to_sql(df,
            table_name = "order_product_edge_select",
            if_exists = "replace",
            primary_index = "ITEM1OF2")

In [ ]:
# Define both edge and node metadata
graph_obj = td_graph_function.td_graph_object(edge_table_name = "order_product_edge_select",
                                              edge_from_node_column_name = edge_from_id,
                                              edge_to_node_column_name = edge_to_id,
                                              edge_weight_column_name = "CONFIDENCE",
                                              weight_type = 'P',
                                              node_table_name = node_table,
                                              node_id_column_name = node_id,
                                              node_label_column_name = 'product_name'
                                             )

In [ ]:
# Identify the importany SKU
df_du, iters_du = graph_obj.td_pagerank(directed = True, 
                                        damping = 0.85,
                                        output_table = "order_product_pr", 
                                        temp_output_table =False)

In [ ]:
# Showing the higest importance node (sku)
DataFrame.from_query("""SELECT
p.product_name, pr.pr_score
FROM order_product_pr pr, products_v p
WHERE pr.node = p.product_id""").sort("pr_score", ascending=False)
                     

In [ ]:
# Create community
df_comm, community = graph_obj.td_louvain_community(directed = True,
                                                    p_threshold = 0.2,                             
                                                    max_iterations = 50,
                                                    p_resolution = 0.5,
                                                    output_table = "order_product_comm", 
                                                    temp_output_table = False)

In [ ]:
# showing the community 
product_name = "Strawberry Banana Yogurt"
SQL = f"""
SELECT p.product_id, p.product_name, c.comm_size, c.pr_score,
(CASE WHEN node = representative_node THEN 'YES' ELSE 'No' END) AS key_node
FROM order_product_comm c, products_v p
WHERE c.node = p.product_id 
AND c.comm_id IN
(SELECT comm_id FROM order_product_comm
WHERE node in
(SELECT product_id
FROM products_v
WHERE product_name = '{product_name}'))"""
DataFrame.from_query(SQL).sort(["key_node","pr_score"], ascending=False)

# Disconnect from database

In [ ]:
remove_context()